In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# OPTIMIZED UniXcoder + DEVIGN - 5 EPOCHS, MULTI-GPU
# Full T4 x2 GPU utilization
# ============================================================

import os, json, torch, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.nn import DataParallel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import platform, psutil
from datetime import datetime
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# FOCAL LOSS - Better for imbalanced data
# ============================================================
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ============================================================
# MULTI-GPU SETUP
# ============================================================
print("="*60)
print("MULTI-GPU CONFIGURATION")
print("="*60)
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Number of GPUs Detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:")
        print(f"  Name: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
        print(f"  Compute: {torch.cuda.get_device_properties(i).major}.{torch.cuda.get_device_properties(i).minor}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
use_multi_gpu = torch.cuda.device_count() > 1

print(f"\n{'='*60}")
print(f"Primary Device: {device}")
print(f"Multi-GPU Training: {'ENABLED' if use_multi_gpu else 'DISABLED'}")
if use_multi_gpu:
    print(f"Training will use ALL {torch.cuda.device_count()} GPUs simultaneously")
    print(f"Effective batch size will be multiplied across GPUs")
print("="*60)

# --------------------
# DATASET PATH
# --------------------
DATASET_PATH = "/kaggle/input/devign-uxc"
print(f"\nDataset Location: {DATASET_PATH}")
print("Available files:", os.listdir(DATASET_PATH))

# ============================================================
# LOAD DEVIGN CSV
# ============================================================
def load_devign_csv(path):
    df = pd.read_csv(path)
    
    # Detect code column
    if "func" in df.columns:
        df = df.rename(columns={"func": "code"})
    elif "code" not in df.columns:
        raise ValueError("No code column found")
    
    if "label" not in df.columns:
        raise ValueError("No label column found")
    
    df = df[["code", "label"]].dropna()
    df["label"] = df["label"].astype(int)
    
    # Remove any invalid labels
    df = df[df["label"].isin([0, 1])]
    
    return df

train_df = load_devign_csv(f"{DATASET_PATH}/devignx_train.csv")
val_df   = load_devign_csv(f"{DATASET_PATH}/Devignx_validation.csv")
test_df  = load_devign_csv(f"{DATASET_PATH}/devignx_test.csv")

print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total samples: {len(train_df) + len(val_df) + len(test_df)}")

print("\nTrain Label Distribution:")
print(train_df["label"].value_counts())
class_0_count = (train_df['label']==0).sum()
class_1_count = (train_df['label']==1).sum()
print(f"\nClass 0 (Non-vulnerable): {class_0_count} ({class_0_count/len(train_df)*100:.2f}%)")
print(f"Class 1 (Vulnerable): {class_1_count} ({class_1_count/len(train_df)*100:.2f}%)")
imbalance_ratio = max(class_0_count, class_1_count) / min(class_0_count, class_1_count)
print(f"Imbalance Ratio: {imbalance_ratio:.2f}:1")

# ============================================================
# ENHANCED CLASS WEIGHTS
# ============================================================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)

# Boost minority class weight (aggressive approach)
minority_class = 1 if class_1_count < class_0_count else 0
class_weights[minority_class] = class_weights[minority_class] * 1.5

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("\n" + "="*60)
print("ENHANCED CLASS WEIGHTS (for imbalance handling):")
print(f"Class 0 weight: {class_weights[0]:.4f}")
print(f"Class 1 weight: {class_weights[1]:.4f} {'← BOOSTED (minority)' if minority_class == 1 else ''}")
print("="*60)

# ============================================================
# TOKENIZER & MODEL CONFIG
# ============================================================
MODEL_NAME = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 512  # Increased for better code context

print(f"\nModel: {MODEL_NAME}")
print(f"Max Sequence Length: {MAX_LEN} tokens")

# ============================================================
# DATASET CLASS
# ============================================================
class DevignDataset(Dataset):
    def __init__(self, df):
        self.codes = df["code"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.codes[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = DevignDataset(train_df)
val_dataset = DevignDataset(val_df)
test_dataset = DevignDataset(test_df)

# ============================================================
# WEIGHTED RANDOM SAMPLER (Balance batches)
# ============================================================
sample_weights = []
for label in train_df["label"]:
    sample_weights.append(class_weights[label].item())

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Batch size scaled for multi-GPU
BASE_BATCH_SIZE = 8
BATCH_SIZE = BASE_BATCH_SIZE * max(torch.cuda.device_count(), 1)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,  # Increased for better data loading
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("\n" + "="*60)
print("DATA LOADER CONFIGURATION")
print("="*60)
print(f"Total Batch Size: {BATCH_SIZE}")
if use_multi_gpu:
    print(f"Batch Size per GPU: {BASE_BATCH_SIZE}")
    print(f"Parallel Processing: {torch.cuda.device_count()} GPUs")
print(f"Batches per Epoch: {len(train_loader)}")
print(f"Weighted Sampling: ENABLED")
print(f"Data Workers: 4 per DataLoader")
print("="*60)

# ============================================================
# MODEL INITIALIZATION WITH MULTI-GPU
# ============================================================
print("\nInitializing UniXcoder model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification"
)

# Wrap with DataParallel for multi-GPU
if use_multi_gpu:
    print(f"\n{'='*60}")
    print(f"🚀 ENABLING MULTI-GPU TRAINING")
    print(f"{'='*60}")
    print(f"Wrapping model with DataParallel...")
    print(f"Model will be replicated across {torch.cuda.device_count()} GPUs")
    print(f"Each GPU will process {BASE_BATCH_SIZE} samples in parallel")
    model = DataParallel(model)
    print(f"✓ Multi-GPU setup complete!")
    print("="*60)

model = model.to(device)

# Ensure all parameters are trainable
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable Parameters: {trainable_params:,}")

# ============================================================
# TRAINING CONFIGURATION - 5 EPOCHS
# ============================================================
EPOCHS = 5  # ✓ Optimized to 5 epochs
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.15

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,
    eps=1e-8
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

criterion = FocalLoss(alpha=class_weights, gamma=2.0)

print("\n" + "="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Loss Function: Focal Loss (gamma=2.0)")
print(f"Optimizer: AdamW (weight_decay=0.01)")
print(f"Epochs: {EPOCHS} ← OPTIMIZED")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"LR Schedule: Linear with warmup")
print(f"Warmup Steps: {warmup_steps} ({WARMUP_RATIO*100:.0f}% of training)")
print(f"Total Training Steps: {total_steps}")
print(f"Gradient Clipping: Max norm = 1.0")
print("="*60)

# ============================================================
# EVALUATION FUNCTION
# ============================================================
def evaluate(model, data_loader, dataset_name="Validation", threshold=0.5):
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    total_loss = 0
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(outputs.logits, dim=1)
            preds = (probs[:, 1] >= threshold).long()

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs[:, 1].cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    avg_loss = total_loss / len(data_loader)
    
    print(f"\n{dataset_name} Results (threshold={threshold:.2f}):")
    print(f"  Loss: {avg_loss:.4f}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall: {rec:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  Predictions → Class 0: {(y_pred==0).sum()}, Class 1: {(y_pred==1).sum()}")
    print(f"  Probability Stats → Min: {y_probs.min():.4f}, Max: {y_probs.max():.4f}, Mean: {y_probs.mean():.4f}")
    
    return acc, prec, rec, f1, avg_loss, y_true, y_pred, y_probs

# ============================================================
# TRAINING LOOP - 5 EPOCHS WITH GPU MONITORING
# ============================================================
print("\n" + "="*60)
print("🚀 STARTING TRAINING (5 EPOCHS)")
print("="*60)

best_val_f1 = 0
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    batch_labels = []
    
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch+1}/{EPOCHS}")
    print("="*60)
    
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        batch_labels.extend(labels.cpu().numpy())

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        
        # Progress reporting (5 times per epoch)
        if (batch_idx + 1) % max(len(train_loader) // 5, 1) == 0:
            avg_loss = total_loss / (batch_idx + 1)
            current_lr = scheduler.get_last_lr()[0]
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e}")
            
            # GPU memory monitoring
            if torch.cuda.is_available():
                for gpu_id in range(torch.cuda.device_count()):
                    mem_allocated = torch.cuda.memory_allocated(gpu_id) / 1e9
                    mem_reserved = torch.cuda.memory_reserved(gpu_id) / 1e9
                    mem_total = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
                    utilization = (mem_allocated / mem_total) * 100
                    print(f"    GPU {gpu_id} Memory: {mem_allocated:.2f}GB / {mem_total:.2f}GB ({utilization:.1f}% utilized)")

    avg_train_loss = total_loss / len(train_loader)
    batch_labels = np.array(batch_labels)
    
    print(f"\n{'─'*60}")
    print(f"Epoch {epoch+1} Summary:")
    print(f"  Average Loss: {avg_train_loss:.4f}")
    print(f"  Batch Distribution → Class 0: {(batch_labels==0).sum()}, Class 1: {(batch_labels==1).sum()}")
    print("─"*60)
    
    # Validation
    val_acc, val_prec, val_rec, val_f1, val_loss, _, _, _ = evaluate(model, val_loader, "Validation", threshold=0.5)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        model_to_save = model.module if use_multi_gpu else model
        torch.save(model_to_save.state_dict(), '/kaggle/working/best_unixcoder_devign.pt')
        print(f"  ✓ NEW BEST MODEL SAVED! F1: {best_val_f1:.4f}")
    else:
        print(f"  Current F1: {val_f1:.4f} (Best: {best_val_f1:.4f} at epoch {best_epoch})")

print("\n" + "="*60)
print(f"✓ TRAINING COMPLETE!")
print(f"Best Validation F1: {best_val_f1:.4f} (Epoch {best_epoch})")
print("="*60)

# Load best model
if use_multi_gpu:
    model.module.load_state_dict(torch.load('/kaggle/working/best_unixcoder_devign.pt'))
else:
    model.load_state_dict(torch.load('/kaggle/working/best_unixcoder_devign.pt'))
print("\n✓ Loaded best model for final evaluation")

# ============================================================
# THRESHOLD TUNING ON VALIDATION SET
# ============================================================
print("\n" + "="*60)
print("FINDING OPTIMAL CLASSIFICATION THRESHOLD")
print("="*60)

model.eval()
val_probs = []
val_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        
        val_probs.extend(probs[:, 1].cpu().numpy())
        val_labels.extend(labels.numpy())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

best_threshold = 0.5
best_threshold_f1 = 0

print("\nTesting thresholds from 0.10 to 0.85...")
thresholds = np.arange(0.1, 0.9, 0.05)
for thresh in thresholds:
    preds = (val_probs >= thresh).astype(int)
    f1 = f1_score(val_labels, preds, zero_division=0)
    
    if f1 > best_threshold_f1:
        best_threshold_f1 = f1
        best_threshold = thresh
        print(f"  → New best: threshold={thresh:.2f}, F1={f1:.4f}")

print(f"\n✓ Optimal Threshold: {best_threshold:.2f} (F1: {best_threshold_f1:.4f})")

# ============================================================
# FINAL TEST EVALUATION
# ============================================================
print("\n" + "="*60)
print(f"FINAL TEST SET EVALUATION")
print(f"Using optimized threshold: {best_threshold:.2f}")
print("="*60)

test_acc, test_prec, test_rec, test_f1, test_loss, y_true, y_pred, y_probs = evaluate(
    model, test_loader, "Test", threshold=best_threshold
)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print("\n" + "="*60)
print("📊 FINAL RESULTS SUMMARY")
print("="*60)
print(f"Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Precision: {test_prec:.4f}")
print(f"Recall   : {test_rec:.4f}")
print(f"F1 Score : {test_f1:.4f}")
print(f"FPR      : {fpr:.4f}")
print(f"FNR      : {fnr:.4f}")
print(f"\nConfusion Matrix:")
print(f"              Predicted")
print(f"              Neg    Pos")
print(f"Actual  Neg  {tn:4d}   {fp:4d}")
print(f"        Pos  {fn:4d}   {tp:4d}")
print("="*60)

# ============================================================
# SAVE COMPREHENSIVE RESULTS
# ============================================================
results = {
    "dataset": "Devign",
    "model": "UniXcoder (microsoft/unixcoder-base)",
    "training_config": {
        "epochs": EPOCHS,
        "best_epoch": best_epoch,
        "batch_size_total": BATCH_SIZE,
        "batch_size_per_gpu": BASE_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_sequence_length": MAX_LEN,
        "loss_function": "Focal Loss (gamma=2.0)",
        "optimizer": "AdamW (weight_decay=0.01)",
        "weighted_sampling": True,
        "optimal_threshold": float(best_threshold),
        "multi_gpu_enabled": use_multi_gpu,
        "num_gpus_used": torch.cuda.device_count() if use_multi_gpu else 1
    },
    "dataset_stats": {
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "test_samples": len(test_df),
        "imbalance_ratio": float(imbalance_ratio),
        "train_class_distribution": {
            "class_0": int(class_0_count),
            "class_1": int(class_1_count)
        }
    },
    "metrics": {
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1_score": float(test_f1),
        "fpr": float(fpr),
        "fnr": float(fnr)
    },
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    },
    "predictions": {
        "class_0_count": int((y_pred==0).sum()),
        "class_1_count": int((y_pred==1).sum())
    },
    "hardware": {
        "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else [],
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
        "pytorch_version": torch.__version__,
        "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
        "os": platform.system()
    },
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

out_path = "/kaggle/working/UniXcoder_Devign_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=4)

# Save predictions
predictions_df = pd.DataFrame({
    'true_label': y_true,
    'predicted_label': y_pred,
    'probability_class_1': y_probs
})
predictions_df.to_csv('/kaggle/working/devign_predictions.csv', index=False)

print(f"\n✅ Results saved to: {out_path}")
print(f"✅ Predictions saved to: /kaggle/working/devign_predictions.csv")

# Print final summary
print("\n" + "="*60)
print("🎉 EXECUTION COMPLETE!")
print("="*60)
if use_multi_gpu:
    print(f"✓ Successfully utilized {torch.cuda.device_count()} GPUs")
print(f"✓ Trained for {EPOCHS} epochs (best: epoch {best_epoch})")
print(f"✓ Final F1 Score: {test_f1:.4f}")
print(f"✓ Optimal threshold: {best_threshold:.2f}")
print("="*60)

MULTI-GPU CONFIGURATION
CUDA Available: True
Number of GPUs Detected: 2

GPU 0:
  Name: Tesla T4
  Memory: 15.64 GB
  Compute: 7.5

GPU 1:
  Name: Tesla T4
  Memory: 15.64 GB
  Compute: 7.5

Primary Device: cuda:0
Multi-GPU Training: ENABLED
Training will use ALL 2 GPUs simultaneously
Effective batch size will be multiplied across GPUs

Dataset Location: /kaggle/input/devign-uxc
Available files: ['Devignx_validation.csv', 'devignx_test.csv', 'devignx_train.csv']

DATASET STATISTICS
Train samples: 19122
Validation samples: 2732
Test samples: 2732
Total samples: 24586

Train Label Distribution:
label
0    10356
1     8766
Name: count, dtype: int64

Class 0 (Non-vulnerable): 10356 (54.16%)
Class 1 (Vulnerable): 8766 (45.84%)
Imbalance Ratio: 1.18:1

ENHANCED CLASS WEIGHTS (for imbalance handling):
Class 0 weight: 0.9232
Class 1 weight: 1.6360 ← BOOSTED (minority)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 ENABLING MULTI-GPU TRAINING
Wrapping model with DataParallel...
Model will be replicated across 2 GPUs
Each GPU will process 8 samples in parallel
✓ Multi-GPU setup complete!


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]


Trainable Parameters: 125,931,266

TRAINING CONFIGURATION
Loss Function: Focal Loss (gamma=2.0)
Optimizer: AdamW (weight_decay=0.01)
Epochs: 5 ← OPTIMIZED
Learning Rate: 2e-05
LR Schedule: Linear with warmup
Warmup Steps: 897 (15% of training)
Total Training Steps: 5980
Gradient Clipping: Max norm = 1.0

🚀 STARTING TRAINING (5 EPOCHS)

EPOCH 1/5
  Batch 239/1196 | Loss: 0.2822 | LR: 5.33e-06
    GPU 0 Memory: 2.10GB / 15.64GB (13.4% utilized)
    GPU 1 Memory: 0.02GB / 15.64GB (0.1% utilized)
  Batch 478/1196 | Loss: 0.2757 | LR: 1.07e-05
    GPU 0 Memory: 2.10GB / 15.64GB (13.4% utilized)
    GPU 1 Memory: 0.02GB / 15.64GB (0.1% utilized)
  Batch 717/1196 | Loss: 0.2734 | LR: 1.60e-05
    GPU 0 Memory: 2.10GB / 15.64GB (13.4% utilized)
    GPU 1 Memory: 0.02GB / 15.64GB (0.1% utilized)
  Batch 956/1196 | Loss: 0.2703 | LR: 1.98e-05
    GPU 0 Memory: 2.10GB / 15.64GB (13.4% utilized)
    GPU 1 Memory: 0.02GB / 15.64GB (0.1% utilized)
  Batch 1195/1196 | Loss: 0.2684 | LR: 1.88e-05
   